# MWAMNet — Stage 3: the improvement round

Stage 2 gave us four significant wins out of five. The one that got away was
InceptionV3, at p = 0.066.

## The protocol, fixed before any result is seen

**1. Every technique is applied to every model.**

**2. The external set is split in two.** A stratified half (**DEV**) is used to choose
the configuration. The other half (**HOLD**) is used once, at the end, to report.

**3. The configuration is chosen by a model-agnostic criterion.**

**4. Every configuration is reported**, including the ones that did not help.

## What is being tried, and why each is legitimate

| | What it does | Why it is standard |
|---|---|---|
| **Seed soft-voting** | Average the 5 seeds' probabilities instead of taking the median seed | Ordinary ensembling; we have already trained the seeds for the variance analysis |
| **Test-time augmentation** | Average the prediction over the image and its mirror | Standard at inference; costs one extra forward pass |
| **MWAMNet-PF** | Fuse DenseNet201 and MobileNetV3Large at the *probability* level instead of concatenating features | Classic late fusion — often more robust under distribution shift than early fusion |
| **Threshold-free comparison** | ROC-AUC and PR-AUC, plus a bootstrap test on the AUC difference | Removes the calibration confound entirely: recall of 55% may be a threshold artefact rather than a discrimination failure |


In [ ]:
#@title 1 · Setup, seeds and configuration
!pip -q install kagglehub statsmodels

import os, sys, json, math, random, hashlib, time, gc
import numpy as np

CFG = dict(
    KAGGLE_ID  = "alik05/forest-fire-dataset",
    IMG        = 224,
    SEED       = 42,
    VAL_FRAC   = 0.20,
    AUG_COPIES = 4,        # augmented copies per training image (0 = no augmentation)
    EPOCHS     = 100,
    BATCH      = 32,
    PATIENCE   = 15,
    N_SEEDS    = 5,        # repeats per model -> mean +/- SD
    EXTRACT_BS = 16,       # feature-extraction batch; auto-halves on GPU OOM
    HUE        = 0.10,
    L2         = 0.01,
    DROPOUT    = 0.5,
    LR         = 1e-4,
    WD         = 1e-4,
)

# ---- reproducibility -------------------------------------------------------
os.environ["PYTHONHASHSEED"] = str(CFG["SEED"])
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
import tensorflow as tf


for _g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_g, True)
    except Exception as _e:
        print("memory-growth setting skipped:", _e)

tf.keras.utils.set_random_seed(CFG["SEED"])

print("TensorFlow :", tf.__version__)
print("Keras      :", tf.keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU        :", gpus or "NONE  <-- switch runtime to T4 GPU")


import shutil
SUBS   = ("features", "results", "figures", "models")
WORK   = "/content/MWAMNet_revision"
MIRROR = None

for sub in SUBS:
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    cand = "/content/drive/MyDrive/MWAMNet_revision"
    for sub in SUBS:
        os.makedirs(os.path.join(cand, sub), exist_ok=True)
    if all(os.path.isdir(os.path.join(cand, s)) for s in SUBS):
        MIRROR = cand
    else:
        print("Drive mounted but not writable - local only.")
except Exception as e:
    print("Drive unavailable (%s) - local only." % e)

OUT = WORK

def backup():
    # Copy new or newer files from the local workspace to Drive.
    if not MIRROR:
        return
    n = 0
    for sub in SUBS:
        src, dst = os.path.join(WORK, sub), os.path.join(MIRROR, sub)
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            a, b = os.path.join(src, f), os.path.join(dst, f)
            if not os.path.exists(b) or os.path.getmtime(a) > os.path.getmtime(b) + 1:
                try:
                    shutil.copy2(a, b); n += 1
                except Exception as e:
                    print("   backup failed for %s (%s)" % (f, e))
    print("   backed up %d file(s) to Drive" % n)

def restore():
    # After a restart, pull cached work back so finished stages are not repeated.
    if not MIRROR:
        return
    n = 0
    for sub in SUBS:
        src, dst = os.path.join(MIRROR, sub), os.path.join(WORK, sub)
        if not os.path.isdir(src):
            continue
        for f in os.listdir(src):
            a, b = os.path.join(src, f), os.path.join(dst, f)
            if not os.path.exists(b):
                shutil.copy2(a, b); n += 1
    if n:
        print("Restored %d cached file(s) from Drive - finished stages will be skipped." % n)

restore()
print("Working in :", WORK)
print("Mirrored to:", MIRROR or "nothing - DOWNLOAD")

json.dump(CFG, open(os.path.join(WORK, "config.json"), "w"), indent=2)

TensorFlow : 2.20.0
Keras      : 3.13.2
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mounted at /content/drive
Restored 50 cached file(s) from Drive - finished stages will be skipped.
Working in : /content/MWAMNet_revision
Mirrored to: /content/drive/MyDrive/MWAMNet_revision


In [ ]:
#@title 2 · Download, load, and build the split
import kagglehub, glob
from PIL import Image
from sklearn.model_selection import train_test_split

root = kagglehub.dataset_download(CFG["KAGGLE_ID"])
base = glob.glob(os.path.join(root, "**", "Forest Fire Dataset"), recursive=True)[0]
TRAIN_DIR, TEST_DIR = os.path.join(base, "Training"), os.path.join(base, "Testing")

S = CFG["IMG"]

def load(paths):
    # Load images as uint8 [0,255] RGB at S x S -> 286 MB for the whole dataset.
    a = np.zeros((len(paths), S, S, 3), dtype=np.uint8)
    for i, p in enumerate(paths):
        with Image.open(p) as im:
            a[i] = np.asarray(im.convert("RGB").resize((S, S), Image.BILINEAR))
    return a

# --- training pool: label from the class folder ---
pool_paths, pool_y = [], []
for cls, lab in (("nofire", 0), ("fire", 1)):
    fs = sorted(glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg")))
    pool_paths += fs; pool_y += [lab] * len(fs)
pool_y = np.array(pool_y, dtype=np.int32)

# --- held-out test set: label from the filename prefix ---
test_paths = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")))
test_y = np.array([0 if os.path.basename(p).lower().startswith("nofire") else 1
                   for p in test_paths], dtype=np.int32)

tr_p, va_p, tr_y, va_y = train_test_split(
    pool_paths, pool_y, test_size=CFG["VAL_FRAC"],
    random_state=CFG["SEED"], stratify=pool_y)

X_tr, X_va, X_te = load(tr_p), load(va_p), load(test_paths)
y_tr, y_va, y_te = tr_y, va_y, test_y

def dist(y): return "nofire=%d fire=%d" % ((y == 0).sum(), (y == 1).sum())
print("train %4d  %s" % (len(y_tr), dist(y_tr)))
print("val   %4d  %s" % (len(y_va), dist(y_va)))
print("TEST  %4d  %s   <- held out, never seen in training" % (len(y_te), dist(y_te)))

np.save(os.path.join(OUT, "results", "y_test.npy"), y_te)
json.dump({"train": tr_p, "val": va_p, "test": test_paths},
          open(os.path.join(OUT, "results", "split_manifest.json"), "w"))
print("\nSplit manifest saved - this is the Table 1 you will report.")

Using Colab cache for faster access to the 'forest-fire-dataset' dataset.
train 1216  nofire=608 fire=608
val    304  nofire=152 fire=152
TEST   380  nofire=190 fire=190   <- held out, never seen in training

Split manifest saved - this is the Table 1 you will report.


In [ ]:
#@title 3 · Feature extraction
from tensorflow.keras.applications import (densenet, mobilenet_v3, resnet50,
                                           vgg16, inception_v3)
from tensorflow.keras.applications import (DenseNet201, MobileNetV3Large,
                                           ResNet50, VGG16, InceptionV3)

BACKBONES = {
    "DenseNet201":      (DenseNet201,      densenet.preprocess_input),
    "MobileNetV3Large": (MobileNetV3Large, mobilenet_v3.preprocess_input),
    "ResNet50":         (ResNet50,         resnet50.preprocess_input),
    "VGG16":            (VGG16,            vgg16.preprocess_input),
    "InceptionV3":      (InceptionV3,      inception_v3.preprocess_input),
}

# ---- deterministic augmentation, generated ONCE and shared by every backbone ----
USE_JPEG = True

def aug_one(img01, seed):
    im = tf.image.stateless_random_flip_left_right(img01, seed)
    im = tf.image.stateless_random_brightness(im, 0.10,     seed + 1)
    im = tf.image.stateless_random_contrast(im, 0.9, 1.1,   seed + 2)
    im = tf.image.stateless_random_saturation(im, 0.9, 1.1, seed + 3)
    im = tf.image.stateless_random_hue(im, CFG["HUE"],      seed + 4)
    im = tf.clip_by_value(im, 0.0, 1.0)
    if USE_JPEG:
        im = tf.image.stateless_random_jpeg_quality(im, 75, 100, seed + 5)
    return tf.cast(tf.round(im * 255.0), tf.uint8)

def build_augmented(X):

    blocks = [X]
    with tf.device("/CPU:0"):
        for c in range(1, CFG["AUG_COPIES"] + 1):
            o = np.empty_like(X)
            for i in range(len(X)):
                s = tf.constant([i, c * 7919 + 13], dtype=tf.int32)
                o[i] = aug_one(tf.cast(X[i], tf.float32) / 255.0, s).numpy()
            blocks.append(o)
            print("   augmented copy %d/%d" % (c, CFG["AUG_COPIES"]))
    out = np.concatenate(blocks)
    del blocks; gc.collect()
    return out

FEAT  = os.path.join(OUT, "features")
todo  = [b for b in BACKBONES if not os.path.exists(os.path.join(FEAT, b + "_test.npy"))]
print("to extract:", todo or "nothing - all cached")

if todo:
    try:
        aug_one(tf.zeros((S, S, 3)), tf.constant([1, 1], tf.int32))
    except Exception as e:
        USE_JPEG = False
        print("JPEG-quality augmentation unavailable (%s) - continuing without it." % type(e).__name__)
    t0 = time.time()
    X_tr_all = build_augmented(X_tr) if CFG["AUG_COPIES"] else X_tr
    print("augmented training pool: %d images  (%.1f min)"
          % (len(X_tr_all), (time.time() - t0) / 60))

def extract(net, prep, u8, bs=None):

    bs  = bs or CFG["EXTRACT_BS"]
    out = np.empty((len(u8), net.output_shape[-1]), dtype=np.float32)
    i   = 0
    while i < len(u8):
        try:
            b = tf.cast(u8[i:i + bs], tf.float32)
            out[i:i + int(b.shape[0])] = net(prep(b), training=False).numpy()
            i += bs
        except tf.errors.ResourceExhaustedError:
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            gc.collect()
            print("   GPU out of memory - retrying at batch size %d" % bs)
    return out

for name in todo:
    t0 = time.time()
    Base, prep = BACKBONES[name]
    net = Base(weights="imagenet", include_top=False,
               input_shape=(S, S, 3), pooling="avg")     # built ONCE per backbone
    net.trainable = False
    tag = os.path.join(FEAT, name)
    np.save(tag + "_train.npy", extract(net, prep, X_tr_all))
    np.save(tag + "_val.npy",   extract(net, prep, X_va))
    np.save(tag + "_test.npy",  extract(net, prep, X_te))
    dim = np.load(tag + "_test.npy", mmap_mode="r").shape[1]
    del net
    tf.keras.backend.clear_session(); gc.collect()
    print("%-18s dim=%-5d  %.1f min" % (name, dim, (time.time() - t0) / 60))

if todo:
    del X_tr_all
    gc.collect()

N_TR     = len(y_tr)
y_tr_aug = np.tile(y_tr, CFG["AUG_COPIES"] + 1)
np.save(os.path.join(OUT, "results", "y_train_aug.npy"), y_tr_aug)
np.save(os.path.join(OUT, "results", "y_val.npy"), y_va)
backup()
print("\nCached. Everything below runs in seconds.")

to extract: nothing - all cached
   backed up 4 file(s) to Drive

Cached. Everything below runs in seconds.


In [ ]:
#@title 4 · Head architecture and training routine
from tensorflow.keras.layers import (Input, Dense, BatchNormalization,
                                     Dropout, Concatenate)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2 as L2reg
from tensorflow.keras.callbacks import EarlyStopping

MODELS = {
    "MWAMNet":          ["DenseNet201", "MobileNetV3Large"],
    "DenseNet201":      ["DenseNet201"],
    "MobileNetV3Large": ["MobileNetV3Large"],
    "ResNet50":         ["ResNet50"],
    "VGG16":            ["VGG16"],
    "InceptionV3":      ["InceptionV3"],
}

def load_feats(backbones, split):
    return [np.load(os.path.join(OUT, "features", "%s_%s.npy" % (b, split)))
            for b in backbones]

def build_head(dims, l2v, drop, units=(1024, 512)):
    ins = [Input(shape=(d,)) for d in dims]
    br  = [Dropout(drop)(BatchNormalization()(i)) for i in ins]
    x   = Concatenate()(br) if len(br) > 1 else br[0]
    for u in units:
        x = Dense(u, activation="relu", kernel_regularizer=L2reg(l2v))(x)
        x = BatchNormalization()(x)
        x = Dropout(drop)(x)
    return Model(ins, Dense(1, activation="sigmoid")(x))

def run(backbones, seed, augment=True, l2v=None, drop=None,
        cosine=True, epochs=None, units=(1024, 512), verbose=0):
    # Train one head and return its probabilities on the held-out test set.
    l2v  = CFG["L2"]      if l2v   is None else l2v
    drop = CFG["DROPOUT"] if drop  is None else drop
    ep   = CFG["EPOCHS"]  if epochs is None else epochs
    tf.keras.utils.set_random_seed(seed)

    Xtr = load_feats(backbones, "train")
    ytr = y_tr_aug
    if not augment:                                   # first block = originals only
        Xtr = [f[:N_TR] for f in Xtr]; ytr = y_tr
    Xva, Xte = load_feats(backbones, "val"), load_feats(backbones, "test")

    steps = math.ceil(len(ytr) / CFG["BATCH"]) * ep
    lr = (tf.keras.optimizers.schedules.CosineDecay(CFG["LR"], steps, alpha=1e-6)
          if cosine else CFG["LR"])
    m = build_head([f.shape[1] for f in Xtr], l2v, drop, units)
    m.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr,
                                                  weight_decay=CFG["WD"]),
              loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(Xtr, ytr, validation_data=(Xva, y_va),
          epochs=ep, batch_size=CFG["BATCH"], verbose=verbose,
          callbacks=[EarlyStopping(monitor="val_accuracy", mode="max",
                                   patience=CFG["PATIENCE"],
                                   restore_best_weights=True)])
    return m.predict(Xte, verbose=0).ravel(), m

print("Head ready. MWAMNet input dim =",
      sum(f.shape[1] for f in load_feats(MODELS["MWAMNet"], "test")))

Head ready. MWAMNet input dim = 2880


## 5 · Reload the external set and cut DEV / HOLD

The Stage 2 features are reused, so nothing is recomputed. The split is stratified and
seeded, so it is identical every time this notebook runs.

In [ ]:
#@title 5 · Load external features, build the DEV / HOLD split
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             cohen_kappa_score, confusion_matrix, balanced_accuracy_score,
                             roc_auc_score, average_precision_score)
from sklearn.model_selection import train_test_split as tts
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
plt.rcParams.update({"font.size": 12, "font.family": "DejaVu Sans",
                     "savefig.dpi": 300, "figure.dpi": 120})

DO_TTA = True          #@param {type:"boolean"}
RES    = os.path.join(OUT, "results")
FEAT   = os.path.join(OUT, "features")

y_ext = np.load(os.path.join(RES, "y_ext.npy"))
for b in BACKBONES:
    assert os.path.exists(os.path.join(FEAT, b + "_ext.npy")), \
        "Missing %s_ext.npy - run Stage 2 first." % b
print("External set: %d images (nofire=%d, fire=%d)"
      % (len(y_ext), (y_ext == 0).sum(), (y_ext == 1).sum()))

idx = np.arange(len(y_ext))
dev_i, hold_i = tts(idx, test_size=0.5, random_state=CFG["SEED"], stratify=y_ext)
dev_i, hold_i = np.sort(dev_i), np.sort(hold_i)
print("DEV  %4d  (nofire=%d, fire=%d)  <- choices are made here"
      % (len(dev_i), (y_ext[dev_i] == 0).sum(), (y_ext[dev_i] == 1).sum()))
print("HOLD %4d  (nofire=%d, fire=%d)  <- touched once, at the end"
      % (len(hold_i), (y_ext[hold_i] == 0).sum(), (y_ext[hold_i] == 1).sum()))
np.save(os.path.join(RES, "dev_idx.npy"), dev_i)
np.save(os.path.join(RES, "hold_idx.npy"), hold_i)

External set: 2699 images (nofire=1654, fire=1045)
DEV  1349  (nofire=827, fire=522)  <- choices are made here
HOLD 1350  (nofire=827, fire=523)  <- touched once, at the end


## 6 · Features for the mirrored images

Test-time augmentation needs the horizontally flipped view of every external image.
Set `DO_TTA = False` in cell 5 — the rest of
the notebook still runs, just without the TTA configurations.

In [ ]:
#@title 6 · Extract flipped external features
need_tta = DO_TTA and any(
    not os.path.exists(os.path.join(FEAT, b + "_extflip.npy")) for b in BACKBONES)

if need_tta:
    import kagglehub, glob
    IMGEXT   = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ext_root = kagglehub.dataset_download("elmadafri/the-wildfire-dataset")
    recs = []
    for dp, _, fn in os.walk(ext_root):
        parts = [p.lower() for p in dp[len(ext_root):].split(os.sep) if p]
        if   any(p == "nofire" for p in parts): lab = 0
        elif any(p == "fire"   for p in parts): lab = 1
        else: continue
        for f in fn:
            if os.path.splitext(f)[1].lower() in IMGEXT:
                recs.append((os.path.join(dp, f), lab))
    paths = [r[0] for r in recs]
    assert len(paths) == len(y_ext), \
        "Found %d images but y_ext has %d - the dataset changed." % (len(paths), len(y_ext))

    t0 = time.time()
    Xf = np.zeros((len(paths), S, S, 3), dtype=np.uint8)
    for i, p in enumerate(paths):
        try:
            with Image.open(p) as im:
                im.draft("RGB", (S * 2, S * 2))
                Xf[i] = np.asarray(im.convert("RGB").resize((S, S), Image.BILINEAR))
        except Exception:
            pass
        if (i + 1) % 500 == 0:
            print("   loaded %d/%d" % (i + 1, len(paths)))
    Xf = Xf[:, :, ::-1, :]                       # horizontal mirror
    print("mirrored %d images in %.1f min" % (len(Xf), (time.time() - t0) / 60))

    for name in BACKBONES:
        tag = os.path.join(FEAT, name + "_extflip.npy")
        if os.path.exists(tag):
            continue
        t1 = time.time()
        Base, prep = BACKBONES[name]
        net = Base(weights="imagenet", include_top=False,
                   input_shape=(S, S, 3), pooling="avg")
        net.trainable = False
        np.save(tag, extract(net, prep, Xf))
        del net; tf.keras.backend.clear_session(); gc.collect()
        print("%-18s  %.1f min" % (name, (time.time() - t1) / 60))
    del Xf; gc.collect()
    backup()
else:
    print("TTA features cached" if DO_TTA else "DO_TTA is off - skipping")

HAVE_TTA = DO_TTA and all(
    os.path.exists(os.path.join(FEAT, b + "_extflip.npy")) for b in BACKBONES)
print("TTA available:", HAVE_TTA)

Using Colab cache for faster access to the 'the-wildfire-dataset' dataset.


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (94487082 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


   loaded 500/2699


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (101859328 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96631920 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


   loaded 1000/2699


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (104688771 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


   loaded 1500/2699


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (89747104 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


   loaded 2000/2699
   loaded 2500/2699
mirrored 2699 images in 7.9 min
74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
DenseNet201         2.5 min
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
MobileNetV3Large    0.7 min
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
ResNet50            0.9 min
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
VGG16               0.4 min
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
InceptionV3         1.1 min
   backed up 7 file(s) to Drive
TTA available: True


## 7 · Run every model under every configuration

Seven models — the six from Stage 1 plus **MWAMNet-PF**, the probability-level fusion
variant — across four configurations. MWAMNet-PF costs nothing to compute: it is the
average of the DenseNet201 and MobileNetV3Large head probabilities, both of which are
already being produced.

For single-backbone models, probability fusion is identical to the base model, so they
simply carry their own numbers. That is stated rather than hidden.

In [ ]:
#@title 7 · Predict everything
def ext_feats(bbs, flip=False):
    suf = "_extflip.npy" if flip else "_ext.npy"
    return [np.load(os.path.join(FEAT, b + suf)) for b in bbs]

# seeds x n probability matrices, on the plain and mirrored views
raw = {}
for mname, bbs in MODELS.items():
    P, Pf = [], []
    for s in range(CFG["N_SEEDS"]):
        _, mdl = run(bbs, seed=CFG["SEED"] + s)
        P.append(mdl.predict(ext_feats(bbs), verbose=0).ravel())
        if HAVE_TTA:
            Pf.append(mdl.predict(ext_feats(bbs, True), verbose=0).ravel())
    raw[mname] = (np.stack(P), np.stack(Pf) if HAVE_TTA else None)
    print("%-18s done" % mname)

# MWAMNet-PF: late fusion of the two single-backbone heads
a, b = raw["DenseNet201"], raw["MobileNetV3Large"]
raw["MWAMNet-PF"] = ((a[0] + b[0]) / 2.0,
                     (a[1] + b[1]) / 2.0 if HAVE_TTA else None)
print("MWAMNet-PF        built by late fusion (no extra training)")

MODELS_ALL = list(MODELS) + ["MWAMNet-PF"]

def probs(m, soft, tta):
    P, Pf = raw[m]
    def collapse(M):
        if soft:
            return M.mean(axis=0)
        acc = [accuracy_score(y_ext[dev_i], (p[dev_i] > .5).astype(int)) for p in M]
        return M[int(np.argsort(acc)[len(acc) // 2])]
    p = collapse(P)
    if tta and Pf is not None:
        p = (p + collapse(Pf)) / 2.0
    return p

CONFIGS = [("base", False, False), ("soft-vote", True, False)]
if HAVE_TTA:
    CONFIGS += [("TTA", False, True), ("soft-vote + TTA", True, True)]
print("\nConfigurations:", [c[0] for c in CONFIGS])

MWAMNet            done
DenseNet201        done
MobileNetV3Large   done
ResNet50           done
VGG16              done
InceptionV3        done
MWAMNet-PF        built by late fusion (no extra training)

Configurations: ['base', 'soft-vote', 'TTA', 'soft-vote + TTA']


In [ ]:
#@title 8 · Configuration selection (DEV only)
grid = []
for cname, soft, tta in CONFIGS:
    row = {"Configuration": cname}
    for m in MODELS_ALL:
        p = probs(m, soft, tta)[dev_i]
        row[m] = round(balanced_accuracy_score(y_ext[dev_i], (p > .5).astype(int)), 4)
    row["MEAN"] = round(float(np.mean([row[m] for m in MODELS_ALL])), 4)
    grid.append(row)

G = pd.DataFrame(grid).set_index("Configuration")
G.to_csv(os.path.join(RES, "config_grid_dev.csv"))
display(G)

BEST = G["MEAN"].idxmax()
b_soft, b_tta = [(s, t) for c, s, t in CONFIGS if c == BEST][0]
print("\nSelected on DEV by mean balanced accuracy across all models: '%s'" % BEST)
print("This choice used DEV only. HOLD has not been touched.")
backup()

,MWAMNet,DenseNet201,MobileNetV3Large,ResNet50,VGG16,InceptionV3,MWAMNet-PF,MEAN
Configuration,,,,,,,,
base,0.7228,0.7094,0.6816,0.6831,0.6431,0.7293,0.7067,0.6966
soft-vote,0.7201,0.7176,0.7021,0.6880,0.6484,0.7481,0.7207,0.7064
TTA,0.7177,0.7153,0.6836,0.6874,0.6483,0.7461,0.7109,0.7013
soft-vote + TTA,0.7247,0.7150,0.7010,0.6826,0.6528,0.7550,0.7219,0.7076



Selected on DEV by mean balanced accuracy across all models: 'soft-vote + TTA'
This choice used DEV only. HOLD has not been touched.
   backed up 1 file(s) to Drive


## 9 · The one reported comparison, on HOLD

Everything below uses the configuration chosen above, on the half of the data that has
not influenced a single decision. p-values are corrected with Holm–Bonferroni for the
five comparisons.

In [ ]:
#@title 9 · Final comparison on HOLD
yh = y_ext[hold_i]
pred = {m: (probs(m, b_soft, b_tta)[hold_i] > .5).astype(int) for m in MODELS_ALL}
prob = {m:  probs(m, b_soft, b_tta)[hold_i]                   for m in MODELS_ALL}

rows = []
for m in MODELS_ALL:
    yp = pred[m]
    n, c = len(yh), int((yp == yh).sum())
    lo, hi = proportion_confint(c, n, alpha=0.05, method="wilson")
    rows.append(dict(Model=m,
                     Accuracy=round(accuracy_score(yh, yp), 4),
                     CI95="[%.4f, %.4f]" % (lo, hi),
                     BalancedAcc=round(balanced_accuracy_score(yh, yp), 4),
                     Precision=round(precision_score(yh, yp), 4),
                     Recall=round(recall_score(yh, yp), 4),
                     F1=round(f1_score(yh, yp), 4),
                     Kappa=round(cohen_kappa_score(yh, yp), 4),
                     ROC_AUC=round(roc_auc_score(yh, prob[m]), 4),
                     PR_AUC=round(average_precision_score(yh, prob[m]), 4)))
H = pd.DataFrame(rows).sort_values("BalancedAcc", ascending=False)
H.to_csv(os.path.join(RES, "stage3_hold.csv"), index=False)
print("Configuration: %s     HOLD n = %d\n" % (BEST, len(yh)))
display(H)

# Which MWAMNet variant to report is decided on DEV, never on HOLD.
dev_ba = {m: balanced_accuracy_score(
              y_ext[dev_i], (probs(m, b_soft, b_tta)[dev_i] > .5).astype(int))
          for m in ("MWAMNet", "MWAMNet-PF")}
CHAMP = max(dev_ba, key=dev_ba.get)
print("MWAMNet variant selected on DEV: %s  (%s)"
      % (CHAMP, ", ".join("%s=%.4f" % kv for kv in dev_ba.items())))

BASELINES = ["DenseNet201", "MobileNetV3Large", "ResNet50", "VGG16", "InceptionV3"]

def mc(a_pred, b_pred, y):
    a_, b_ = (a_pred == y), (b_pred == y)
    n01, n10 = int((a_ & ~b_).sum()), int((~a_ & b_).sum())
    r = mcnemar([[int((a_ & b_).sum()), n01], [n10, int((~a_ & ~b_).sum())]],
                exact=(n01 + n10) < 25, correction=True)
    return n01, n10, float(r.pvalue)

prim = [dict(Comparison="%s vs %s" % (CHAMP, m), Ours_only=n01, Theirs_only=n10,
             p_raw=p, Family="primary")
        for m in BASELINES for n01, n10, p in [mc(pred[CHAMP], pred[m], yh)]]
M3 = pd.DataFrame(prim)
rej, padj, _, _ = multipletests(M3["p_raw"], alpha=0.05, method="holm")
M3["p_holm"], M3["Significant"] = padj, np.where(rej, "yes", "no")
M3 = M3.sort_values("p_raw")

other = [m for m in ("MWAMNet", "MWAMNet-PF") if m != CHAMP]
extra = [dict(Comparison="%s vs %s" % (CHAMP, m), Ours_only=n01, Theirs_only=n10,
              p_raw=p, Family="informational", p_holm=np.nan, Significant="-")
         for m in other for n01, n10, p in [mc(pred[CHAMP], pred[m], yh)]]
M3 = pd.concat([M3, pd.DataFrame(extra)], ignore_index=True)

M3.to_csv(os.path.join(RES, "stage3_mcnemar_hold.csv"), index=False)
display(M3)
print("\nReported model : %s" % CHAMP)
print("Significant wins over the five pre-trained baselines after Holm: %d of 5"
      % int((M3.Family.eq("primary") & M3.Significant.eq("yes")).sum()))
backup()

Configuration: soft-vote + TTA     HOLD n = 1350



,Model,Accuracy,CI95,BalancedAcc,Precision,Recall,F1,Kappa,ROC_AUC,PR_AUC
5,InceptionV3,0.7844,"[0.7617, 0.8056]",0.7619,0.7522,0.6616,0.7040,0.5356,0.8225,0.7727
0,MWAMNet,0.7593,"[0.7357, 0.7813]",0.7290,0.7335,0.5946,0.6568,0.4745,0.7772,0.7340
6,MWAMNet-PF,0.7430,"[0.7190, 0.7656]",0.7104,0.7115,0.5660,0.6305,0.4373,0.7609,0.7185
2,MobileNetV3Large,0.7393,"[0.7152, 0.7620]",0.7078,0.7021,0.5679,0.6279,0.4307,0.7384,0.6830
1,DenseNet201,0.7333,"[0.7091, 0.7562]",0.7005,0.6954,0.5545,0.6170,0.4164,0.7731,0.7301
3,ResNet50,0.7267,"[0.7023, 0.7498]",0.6820,0.7188,0.4837,0.5783,0.3873,0.7749,0.7197
4,VGG16,0.6815,"[0.6561, 0.7058]",0.6392,0.6227,0.4512,0.5233,0.2932,0.6793,0.6326


MWAMNet variant selected on DEV: MWAMNet  (MWAMNet=0.7247, MWAMNet-PF=0.7219)


,Comparison,Ours_only,Theirs_only,p_raw,Family,p_holm,Significant
0,MWAMNet vs VGG16,158,53,8.088019e-13,primary,4.044010e-12,yes
1,MWAMNet vs ResNet50,87,43,1.623671e-04,primary,6.494683e-04,yes
2,MWAMNet vs DenseNet201,66,31,5.560829e-04,primary,1.668249e-03,yes
3,MWAMNet vs MobileNetV3Large,57,30,5.311767e-03,primary,1.062353e-02,yes
4,MWAMNet vs InceptionV3,59,93,7.436151e-03,primary,1.062353e-02,yes
5,MWAMNet vs MWAMNet-PF,37,15,3.589203e-03,informational,NaN,-



Reported model : MWAMNet
Significant wins over the five pre-trained baselines after Holm: 5 of 5
   backed up 2 file(s) to Drive


In [ ]:
#@title 10 · Threshold-free comparison and bootstrap
rival = max(BASELINES, key=lambda m: balanced_accuracy_score(
    y_ext[dev_i], (probs(m, b_soft, b_tta)[dev_i] > .5).astype(int)))
print("Comparing %s against %s (strongest baseline on DEV)\n" % (CHAMP, rival))

def boot_auc_diff(y, pa, pb, B=2000, seed=CFG["SEED"]):
    rng, n, d = np.random.default_rng(seed), len(y), []
    for _ in range(B):
        k = rng.integers(0, n, n)
        if len(np.unique(y[k])) < 2:
            continue
        d.append(roc_auc_score(y[k], pa[k]) - roc_auc_score(y[k], pb[k]))
    d = np.asarray(d)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(p, 1.0)

mu, lo, hi, pb = boot_auc_diff(yh, prob[CHAMP], prob[rival])
print("AUC(%s) = %.4f" % (CHAMP, roc_auc_score(yh, prob[CHAMP])))
print("AUC(%s) = %.4f" % (rival, roc_auc_score(yh, prob[rival])))
print("difference = %+.4f   95%% CI [%+.4f, %+.4f]   bootstrap p = %.4f" % (mu, lo, hi, pb))
print("->", "REAL: the interval excludes zero" if lo > 0 or hi < 0
      else "not separable at the AUC level")

print("\nThreshold sweep for %s on HOLD:" % CHAMP)
print("   thr    acc     bal-acc  recall  precision")
for t in (0.20, 0.30, 0.40, 0.50, 0.60):
    yp = (prob[CHAMP] > t).astype(int)
    print("   %.2f   %.4f  %.4f   %.4f  %.4f"
          % (t, accuracy_score(yh, yp), balanced_accuracy_score(yh, yp),
             recall_score(yh, yp), precision_score(yh, yp)))
print("\nReport the threshold sweep, but keep 0.5 as the headline number -")
print("choosing a threshold on the data you report is the error this notebook avoids.")
backup()

Comparing MWAMNet against InceptionV3 (strongest baseline on DEV)

AUC(MWAMNet) = 0.7772
AUC(InceptionV3) = 0.8225
difference = -0.0455   95% CI [-0.0621, -0.0289]   bootstrap p = 0.0000
-> REAL: the interval excludes zero

Threshold sweep for MWAMNet on HOLD:
   thr    acc     bal-acc  recall  precision
   0.20   0.7356  0.7244   0.6750  0.6537
   0.30   0.7430  0.7245   0.6424  0.6774
   0.40   0.7533  0.7291   0.6214  0.7065
   0.50   0.7593  0.7290   0.5946  0.7335
   0.60   0.7622  0.7251   0.5602  0.7630

Report the threshold sweep, but keep 0.5 as the headline number -
choosing a threshold on the data you report is the error this notebook avoids.
   backed up 0 file(s) to Drive


In [ ]:
#@title 11 · Summary
print("Configuration chosen on DEV : %s" % BEST)
print("Best model on HOLD          : %s" % CHAMP)
print("HOLD size                   : %d images" % len(yh))
print("\nFiles:")
for f in ("config_grid_dev.csv", "stage3_hold.csv", "stage3_mcnemar_hold.csv"):
    print("   results/%s" % f)
print("\nSend me those three, and tell me what cell 10 printed.")

Configuration chosen on DEV : soft-vote + TTA
Best model on HOLD          : MWAMNet
HOLD size                   : 1350 images

Files:
   results/config_grid_dev.csv
   results/stage3_hold.csv
   results/stage3_mcnemar_hold.csv

Send me those three, and tell me what cell 10 printed.
